In [12]:
# ============================================================
# UCI HAR - 5 Model Comparison
#
# 1. ResNet
# 2. SqueezeNet
# 3. Inception
# 4. Depthwise Separable CNN
# 5. CNN + Positional Encoding + Transformer
#
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import os
import zipfile
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    DepthwiseConv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    BatchNormalization,
    ReLU,
    Add,
    Concatenate,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    Embedding
)

from tensorflow.keras.models import Model

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 0-1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 0-2. UCI-HAR Path
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "UCI HAR Dataset"
)


# ============================================================
# 0-3. Download UCI-HAR
# ============================================================

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/00240/"
        "UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        z.extractall(
            "/content/drive/MyDrive/Colab Notebooks"
        )

    print("Download completed")


# ============================================================
# 0-4. Signal Names
# ============================================================

signals = [

    "body_acc_x",
    "body_acc_y",
    "body_acc_z",

    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",

    "total_acc_x",
    "total_acc_y",
    "total_acc_z"

]


# ============================================================
# 0-5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/"
            f"{signal}_{split}.txt"
        )

        data = np.loadtxt(
            file_path
        )

        X.append(data)


    # --------------------------------------------------------
    # (9, N, 128)
    #        ↓
    # (N, 128, 9)
    # --------------------------------------------------------

    X = np.stack(
        X,
        axis=-1
    )


    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    y_path = (
        f"{DATA_PATH}/{split}/"
        f"y_{split}.txt"
    )

    y = np.loadtxt(
        y_path
    ).astype(int)


    # --------------------------------------------------------
    # 1~6 → 0~5
    # --------------------------------------------------------

    y = y - 1


    return X, y


# ============================================================
# 0-6. Load Train / Test
# ============================================================

X_train, y_train = load_data(
    "train"
)

X_test, y_test = load_data(
    "test"
)


print("\n")
print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_test :",
    X_test.shape
)

print(
    "y_test :",
    y_test.shape
)


# ============================================================
# 0-7. Train / Validation Split
# ============================================================
#
# Official UCI-HAR train set
#          ↓
#     80% Train
#     20% Validation
#
# Official UCI-HAR test set
#          ↓
#       Test
#
# stratify=y_train
# → 클래스 비율을 유지하면서 분리
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(

    X_train,

    y_train,

    test_size=0.2,

    random_state=SEED,

    stratify=y_train,

    shuffle=True

)


print("\n")
print("=" * 70)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_val  :",
    X_val.shape
)

print(
    "y_val  :",
    y_val.shape
)

print(
    "X_test :",
    X_test.shape
)

print(
    "y_test :",
    y_test.shape
)


# ============================================================
# 0-8. Normalization
# ============================================================
#
# IMPORTANT
#
# mean / std는 training data만 사용해서 계산
#
# validation / test에는
# training mean / std를 그대로 사용
#
# → Data Leakage 방지
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)


X_train = (
    X_train - mean
) / (
    std + 1e-8
)


X_val = (
    X_val - mean
) / (
    std + 1e-8
)


X_test = (
    X_test - mean
) / (
    std + 1e-8
)


print("\n")
print("=" * 70)
print("NORMALIZATION")
print("=" * 70)

print(
    "Training mean shape:",
    mean.shape
)

print(
    "Training std shape:",
    std.shape
)


# ============================================================
# 0-9. One-hot Encoding
# ============================================================

n_outputs = 6


y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)


y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)


y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 0-10. Class Names
# ============================================================

class_names = [

    "WALKING",

    "WALKING_UPSTAIRS",

    "WALKING_DOWNSTAIRS",

    "SITTING",

    "STANDING",

    "LAYING"

]

Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset


ORIGINAL DATASET
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


TRAIN / VALIDATION / TEST SPLIT
X_train: (5881, 128, 9)
y_train: (5881,)
X_val  : (1471, 128, 9)
y_val  : (1471,)
X_test : (2947, 128, 9)
y_test : (2947,)


NORMALIZATION
Training mean shape: (1, 1, 9)
Training std shape: (1, 1, 9)


In [13]:
# ################################################################
# ################################################################
# 1. RESNET
# ################################################################
# ################################################################


# ============================================================
# 1-1. ResNet Block
# ============================================================

def residual_block(
    x,
    filters,
    kernel_size=3
):

    shortcut = x


    # --------------------------------------------------------
    # Main Path - Conv 1
    # --------------------------------------------------------

    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)

    x = ReLU()(x)


    # --------------------------------------------------------
    # Main Path - Conv 2
    # --------------------------------------------------------

    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)


    # --------------------------------------------------------
    # Shortcut Channel Matching
    # --------------------------------------------------------

    if shortcut.shape[-1] != filters:

        shortcut = Conv1D(
            filters,
            1,
            padding="same",
            use_bias=False
        )(shortcut)

        shortcut = BatchNormalization()(
            shortcut
        )


    # --------------------------------------------------------
    # Skip Connection
    # --------------------------------------------------------

    x = Add()([
        x,
        shortcut
    ])

    x = ReLU()(x)


    return x


# ============================================================
# 1-2. Build ResNet
# ============================================================

def build_resnet():

    inputs = Input(
        shape=(128, 9)
    )


    x = Conv1D(
        32,
        3,
        padding="same",
        use_bias=False
    )(inputs)

    x = BatchNormalization()(x)

    x = ReLU()(x)


    x = residual_block(
        x,
        32
    )

    x = MaxPooling1D(
        2
    )(x)


    x = residual_block(
        x,
        64
    )

    x = MaxPooling1D(
        2
    )(x)


    x = residual_block(
        x,
        128
    )


    x = GlobalAveragePooling1D()(
        x
    )


    x = Dense(
        128,
        activation="relu"
    )(x)


    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    return Model(
        inputs,
        outputs,
        name="ResNet"
    )

In [14]:
# ################################################################
# ################################################################
# 2. SQUEEZENET
# ################################################################
# ################################################################


# ============================================================
# 2-1. Fire Module
# ============================================================

def fire_module(
    x,
    squeeze_filters,
    expand_filters
):

    # --------------------------------------------------------
    # Squeeze 1x1
    # --------------------------------------------------------

    squeeze = Conv1D(
        squeeze_filters,
        1,
        padding="same",
        activation="relu"
    )(x)


    # --------------------------------------------------------
    # Expand 1x1
    # --------------------------------------------------------

    expand_1x1 = Conv1D(
        expand_filters,
        1,
        padding="same",
        activation="relu"
    )(squeeze)


    # --------------------------------------------------------
    # Expand 3x3
    # --------------------------------------------------------

    expand_3x3 = Conv1D(
        expand_filters,
        3,
        padding="same",
        activation="relu"
    )(squeeze)


    # --------------------------------------------------------
    # Concatenate
    # --------------------------------------------------------

    x = Concatenate()([
        expand_1x1,
        expand_3x3
    ])


    return x


# ============================================================
# 2-2. Build SqueezeNet
# ============================================================

def build_squeezenet():

    inputs = Input(
        shape=(128, 9)
    )


    x = Conv1D(
        32,
        3,
        padding="same",
        activation="relu"
    )(inputs)


    x = fire_module(
        x,
        squeeze_filters=16,
        expand_filters=32
    )

    x = MaxPooling1D(
        2
    )(x)


    x = fire_module(
        x,
        squeeze_filters=32,
        expand_filters=64
    )

    x = MaxPooling1D(
        2
    )(x)


    x = fire_module(
        x,
        squeeze_filters=64,
        expand_filters=128
    )


    x = GlobalAveragePooling1D()(
        x
    )


    x = Dense(
        128,
        activation="relu"
    )(x)


    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    return Model(
        inputs,
        outputs,
        name="SqueezeNet"
    )

In [15]:
# ################################################################
# ################################################################
# 3. INCEPTION
# ################################################################
# ################################################################


# ============================================================
# 3-1. Inception Block
# ============================================================

def inception_block(
    x,
    filters
):

    # --------------------------------------------------------
    # 1x1 Branch
    # --------------------------------------------------------

    branch1 = Conv1D(
        filters,
        1,
        padding="same",
        activation="relu"
    )(x)


    # --------------------------------------------------------
    # 3x3 Branch
    # --------------------------------------------------------

    branch3 = Conv1D(
        filters,
        3,
        padding="same",
        activation="relu"
    )(x)


    # --------------------------------------------------------
    # 5x5 Branch
    # --------------------------------------------------------

    branch5 = Conv1D(
        filters,
        5,
        padding="same",
        activation="relu"
    )(x)


    # --------------------------------------------------------
    # Pooling Branch
    # --------------------------------------------------------

    branch_pool = MaxPooling1D(
        pool_size=3,
        strides=1,
        padding="same"
    )(x)


    branch_pool = Conv1D(
        filters,
        1,
        padding="same",
        activation="relu"
    )(branch_pool)


    # --------------------------------------------------------
    # Concatenate
    # --------------------------------------------------------

    x = Concatenate()([
        branch1,
        branch3,
        branch5,
        branch_pool
    ])


    return x


# ============================================================
# 3-2. Build Inception
# ============================================================

def build_inception():

    inputs = Input(
        shape=(128, 9)
    )


    x = Conv1D(
        32,
        3,
        padding="same",
        activation="relu"
    )(inputs)


    x = inception_block(
        x,
        32
    )

    x = MaxPooling1D(
        2
    )(x)


    x = inception_block(
        x,
        64
    )

    x = MaxPooling1D(
        2
    )(x)


    x = inception_block(
        x,
        128
    )


    x = GlobalAveragePooling1D()(
        x
    )


    x = Dense(
        128,
        activation="relu"
    )(x)


    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    return Model(
        inputs,
        outputs,
        name="Inception"
    )

In [16]:
# ################################################################
# ################################################################
# 4. DEPTHWISE SEPARABLE CNN
# ################################################################
# ################################################################


# ============================================================
# 4-1. Build Depthwise CNN
# ============================================================

def build_depthwise():

    inputs = Input(
        shape=(128, 9)
    )


    # --------------------------------------------------------
    # Block 1
    # --------------------------------------------------------

    x = Conv1D(
        32,
        3,
        padding="same",
        activation="relu"
    )(inputs)


    x = DepthwiseConv1D(
        3,
        padding="same",
        activation="relu"
    )(x)


    x = Conv1D(
        64,
        1,
        padding="same",
        activation="relu"
    )(x)


    x = MaxPooling1D(
        2
    )(x)


    # --------------------------------------------------------
    # Block 2
    # --------------------------------------------------------

    x = DepthwiseConv1D(
        3,
        padding="same",
        activation="relu"
    )(x)


    x = Conv1D(
        128,
        1,
        padding="same",
        activation="relu"
    )(x)


    x = MaxPooling1D(
        2
    )(x)


    # --------------------------------------------------------
    # Block 3
    # --------------------------------------------------------

    x = DepthwiseConv1D(
        3,
        padding="same",
        activation="relu"
    )(x)


    x = Conv1D(
        128,
        1,
        padding="same",
        activation="relu"
    )(x)


    x = GlobalAveragePooling1D()(
        x
    )


    x = Dense(
        128,
        activation="relu"
    )(x)


    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    return Model(
        inputs,
        outputs,
        name="Depthwise"
    )

In [17]:
# ################################################################
# ################################################################
# 5. CNN + POSITIONAL ENCODING + TRANSFORMER
# ################################################################
# ################################################################


# ============================================================
# 5-1. Positional Embedding
# ============================================================

class PositionalEmbedding(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        sequence_length,
        d_model
    ):

        super().__init__()


        self.position_embedding = Embedding(
            input_dim=sequence_length,
            output_dim=d_model
        )


    def call(self, x):

        positions = tf.range(
            start=0,
            limit=tf.shape(x)[1],
            delta=1
        )


        position_embeddings = (
            self.position_embedding(
                positions
            )
        )


        return (
            x + position_embeddings
        )


# ============================================================
# 5-2. Transformer Encoder
# ============================================================

def transformer_encoder(
    inputs,
    head_size=64,
    num_heads=4,
    ff_dim=128,
    dropout=0.1
):

    # --------------------------------------------------------
    # Layer Normalization
    # --------------------------------------------------------

    x = LayerNormalization(
        epsilon=1e-6
    )(inputs)


    # --------------------------------------------------------
    # Multi-Head Self Attention
    # --------------------------------------------------------

    attention_output = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(
        x,
        x
    )


    attention_output = Dropout(
        dropout
    )(attention_output)


    # --------------------------------------------------------
    # Residual Connection
    # --------------------------------------------------------

    x = Add()([
        inputs,
        attention_output
    ])


    # --------------------------------------------------------
    # Feed Forward Network
    # --------------------------------------------------------

    y = LayerNormalization(
        epsilon=1e-6
    )(x)


    y = Dense(
        ff_dim,
        activation="gelu"
    )(y)


    y = Dropout(
        dropout
    )(y)


    y = Dense(
        inputs.shape[-1]
    )(y)


    y = Dropout(
        dropout
    )(y)


    # --------------------------------------------------------
    # Residual Connection
    # --------------------------------------------------------

    outputs = Add()([
        x,
        y
    ])


    return outputs


# ============================================================
# 5-3. Build CNN + Transformer
# ============================================================

def build_cnn_transformer():

    inputs = Input(
        shape=(128, 9)
    )


    # --------------------------------------------------------
    # CNN Feature Extraction
    # --------------------------------------------------------

    x = Conv1D(
        32,
        3,
        activation="relu"
    )(inputs)


    x = MaxPooling1D(
        2
    )(x)


    x = Conv1D(
        64,
        3,
        activation="relu"
    )(x)


    x = MaxPooling1D(
        2
    )(x)


    # --------------------------------------------------------
    # CNN Output
    #
    # (128, 9)
    #      ↓
    # (126, 32)
    #      ↓
    # (63, 32)
    #      ↓
    # (61, 64)
    #      ↓
    # (30, 64)
    # --------------------------------------------------------


    # --------------------------------------------------------
    # Positional Encoding
    # --------------------------------------------------------

    x = PositionalEmbedding(
        sequence_length=30,
        d_model=64
    )(x)


    # --------------------------------------------------------
    # Transformer Encoder
    # --------------------------------------------------------

    x = transformer_encoder(
        x,
        head_size=64,
        num_heads=4,
        ff_dim=128,
        dropout=0.1
    )


    # --------------------------------------------------------
    # Global Average Pooling
    # --------------------------------------------------------

    x = GlobalAveragePooling1D()(
        x
    )


    # --------------------------------------------------------
    # Dense
    # --------------------------------------------------------

    x = Dense(
        128,
        activation="relu"
    )(x)


    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    outputs = Dense(
        6,
        activation="softmax"
    )(x)


    return Model(
        inputs,
        outputs,
        name="CNN_Transformer"
    )

In [18]:
# ################################################################
# ################################################################
# 6. MODEL TRAINING & EVALUATION
# ################################################################
# ################################################################


# ============================================================
# 6-1. Model List
# ============================================================

model_builders = {

    "ResNet":
        build_resnet,

    "SqueezeNet":
        build_squeezenet,

    "Inception":
        build_inception,

    "Depthwise":
        build_depthwise,

    "CNN_Transformer":
        build_cnn_transformer

}


# ============================================================
# 6-2. Results
# ============================================================

results = []


# ============================================================
# 6-3. Train All Models
# ============================================================

for model_name, build_model in model_builders.items():

    print("\n")
    print("=" * 70)
    print(
        f"TRAINING: {model_name}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Reset Seed
    # --------------------------------------------------------

    np.random.seed(
        SEED
    )

    tf.random.set_seed(
        SEED
    )


    # --------------------------------------------------------
    # Build Model
    # --------------------------------------------------------

    model = build_model()


    # --------------------------------------------------------
    # Parameters
    # --------------------------------------------------------

    params = model.count_params()


    print(
        f"\nParameters: {params:,}"
    )


    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(

        loss="categorical_crossentropy",

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        metrics=[
            "accuracy"
        ]

    )


    # --------------------------------------------------------
    # Callbacks
    # --------------------------------------------------------

    callbacks = [

        EarlyStopping(

            monitor="val_loss",

            patience=5,

            restore_best_weights=True,

            verbose=1

        ),

        ReduceLROnPlateau(

            monitor="val_loss",

            factor=0.5,

            patience=2,

            min_lr=1e-6,

            verbose=1

        )

    ]


    # --------------------------------------------------------
    # Model Summary
    # --------------------------------------------------------

    model.summary()


    # --------------------------------------------------------
    # Start Training
    # --------------------------------------------------------

    start_time = time.time()


    history = model.fit(

        X_train,

        y_train_onehot,

        epochs=50,

        batch_size=64,

        validation_data=(

            X_val,

            y_val_onehot

        ),

        callbacks=callbacks,

        shuffle=True,

        verbose=1

    )


    # --------------------------------------------------------
    # Training Time
    # --------------------------------------------------------

    training_time = (
        time.time()
        -
        start_time
    )


    # --------------------------------------------------------
    # Best Validation Metrics
    # --------------------------------------------------------

    best_val_accuracy = max(
        history.history[
            "val_accuracy"
        ]
    )


    best_val_loss = min(
        history.history[
            "val_loss"
        ]
    )


    actual_epochs = len(
        history.history[
            "loss"
        ]
    )


    # ========================================================
    # Test Prediction
    # ========================================================

    y_prob = model.predict(

        X_test,

        verbose=0

    )


    y_pred = np.argmax(

        y_prob,

        axis=1

    )


    # ========================================================
    # Test Metrics
    # ========================================================

    accuracy = accuracy_score(

        y_test,

        y_pred

    )


    precision = precision_score(

        y_test,

        y_pred,

        average="macro",

        zero_division=0

    )


    recall = recall_score(

        y_test,

        y_pred,

        average="macro",

        zero_division=0

    )


    f1 = f1_score(

        y_test,

        y_pred,

        average="macro",

        zero_division=0

    )


    # ========================================================
    # Save Results
    # ========================================================

    results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1-score":
            f1,

        "Best Val Accuracy":
            best_val_accuracy,

        "Best Val Loss":
            best_val_loss,

        "Epochs":
            actual_epochs,

        "Parameters":
            params,

        "Training Time (sec)":
            training_time

    })


    # ========================================================
    # Result
    # ========================================================

    print("\n")
    print("-" * 60)

    print(
        f"{model_name} RESULT"
    )

    print("-" * 60)


    print(
        f"Best Val Accuracy : "
        f"{best_val_accuracy:.4f}"
    )


    print(
        f"Best Val Loss     : "
        f"{best_val_loss:.4f}"
    )


    print(
        f"Epochs            : "
        f"{actual_epochs}"
    )


    print(
        f"Test Accuracy     : "
        f"{accuracy:.4f}"
    )


    print(
        f"Test Precision    : "
        f"{precision:.4f}"
    )


    print(
        f"Test Recall       : "
        f"{recall:.4f}"
    )


    print(
        f"Test F1-score     : "
        f"{f1:.4f}"
    )


    print(
        f"Parameters        : "
        f"{params:,}"
    )


    print(
        f"Training Time     : "
        f"{training_time:.2f} sec"
    )


    # ========================================================
    # Classification Report
    # ========================================================

    print("\n")
    print(
        "Classification Report"
    )


    print(

        classification_report(

            y_test,

            y_pred,

            target_names=class_names,

            digits=4,

            zero_division=0

        )

    )


    # ========================================================
    # Confusion Matrix
    # ========================================================

    cm = confusion_matrix(

        y_test,

        y_pred

    )


    print("\n")
    print(
        "Confusion Matrix"
    )


    print(cm)


    # ========================================================
    # Clear Session
    # ========================================================

    tf.keras.backend.clear_session()



TRAINING: ResNet

Parameters: 129,382


Model: "ResNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 128, 32)   │        864 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 128, 32)   │      3,072 │ re_lu_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 32)   │          0 │ batch_normalizat… │
│                     │                   │            │ re_lu[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 128, 32)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 64, 32)    │          0 │ re_lu_2[0][0]     │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 64, 64)    │      6,144 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 64, 64)    │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 64, 64)    │     12,288 │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 64, 64)    │      2,048 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64)    │        256 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ batch_normalizat

 Total params: 129,382 (505.40 KB)

 Trainable params: 128,038 (500.15 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 126, 32)   │        896 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 63, 32)    │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 61, 64)    │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 30, 64)    │          0 │ conv1d_1[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 30, 64)    │      1,920 │ max_pooling1d_1[… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 30, 64)    │        128 │ positional_embed… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 30, 64)    │          0 │ positional_embed… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 128)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 30, 64)    │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 30, 64)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 30, 64)    │          0 │ add[0][0],        │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ add_1[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │      8,320 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 6)         │        774 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 101,318 (395.77 KB)

 Trainable params: 101,318 (395.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.8633 - loss: 0.3557 - val_accuracy: 0.9429 - val_loss: 0.1444 - learning_rate: 0.0010
Epoch 2/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9493 - loss: 0.1186 - val_accuracy: 0.9579 - val_loss: 0.1229 - learning_rate: 0.0010
Epoch 3/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9537 - loss: 0.1081 - val_accuracy: 0.9538 - val_loss: 0.1509 - learning_rate: 0.0010
Epoch 4/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9587 - loss: 0.0942 - val_accuracy: 0.9660 - val_loss: 0.0905 - learning_rate: 0.0010
Epoch 5/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9626 - loss: 0.0814 - val_accuracy: 0.9667 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 6/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9636 - loss: 0.0757 - val_accuracy: 0.9701 - val_loss: 0.0846 - learning_rate: 0.0010
Epoch 7/50
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9687 - loss: 0.0682 - val_accurac

In [19]:
# ################################################################
# ################################################################
# 7. FINAL COMPARISON
# ################################################################
# ################################################################


# ============================================================
# 7-1. DataFrame
# ============================================================

results_df = pd.DataFrame(
    results
)


print("\n")
print("=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)


print(

    results_df.to_string(
        index=False
    )

)


# ============================================================
# 7-2. Accuracy / F1
# ============================================================

print("\n")
print("=" * 70)
print("ACCURACY & F1 COMPARISON")
print("=" * 70)


print(

    results_df[
        [
            "Model",
            "Accuracy",
            "F1-score",
            "Best Val Accuracy"
        ]
    ].to_string(
        index=False
    )

)


# ============================================================
# 7-3. Parameter Comparison
# ============================================================

print("\n")
print("=" * 70)
print("PARAMETER COMPARISON")
print("=" * 70)


print(

    results_df[
        [
            "Model",
            "Parameters"
        ]
    ].to_string(
        index=False
    )

)


# ============================================================
# 7-4. Training Time Comparison
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING TIME COMPARISON")
print("=" * 70)


print(

    results_df[
        [
            "Model",
            "Training Time (sec)",
            "Epochs"
        ]
    ].to_string(
        index=False
    )

)


# ============================================================
# 7-5. Final Results Rounded
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS - ROUNDED")
print("=" * 70)


display_df = results_df.copy()


display_df["Accuracy"] = (
    display_df["Accuracy"]
    .round(4)
)


display_df["Precision"] = (
    display_df["Precision"]
    .round(4)
)


display_df["Recall"] = (
    display_df["Recall"]
    .round(4)
)


display_df["F1-score"] = (
    display_df["F1-score"]
    .round(4)
)


display_df["Best Val Accuracy"] = (
    display_df["Best Val Accuracy"]
    .round(4)
)


display_df["Best Val Loss"] = (
    display_df["Best Val Loss"]
    .round(4)
)


display_df["Training Time (sec)"] = (
    display_df["Training Time (sec)"]
    .round(2)
)


print(
    display_df.to_string(
        index=False
    )
)



FINAL MODEL COMPARISON
          Model  Accuracy  Precision   Recall  F1-score  Best Val Accuracy  Best Val Loss  Epochs  Parameters  Training Time (sec)
         ResNet  0.922294   0.922969 0.924778  0.923599           0.982325       0.039642      50      129382            53.283351
     SqueezeNet  0.895487   0.896237 0.896721  0.895406           0.990483       0.025884      50       88886            41.335860
      Inception  0.931116   0.931948 0.932291  0.931675           0.997281       0.013187      30      488070            38.284455
      Depthwise  0.904988   0.904004 0.906006  0.904421           0.983005       0.048584      50       46022            46.015432
CNN_Transformer  0.921276   0.921703 0.921742  0.921552           0.980286       0.052084      18      101318            30.610349


ACCURACY & F1 COMPARISON
          Model  Accuracy  F1-score  Best Val Accuracy
         ResNet  0.922294  0.923599           0.982325
     SqueezeNet  0.895487  0.895406           0.9904